# Signum Demo

**Professional financial charting, inspired by Lightweight Charts.**

Works seamlessly in Jupyter notebooks, Dash apps, and Streamlit dashboards — same chart object, multiple outputs.

Themes: `dark` · `light` · `ft` (Financial Times) · `midnight`

In [1]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "signum" else os.getcwd())

import pandas as pd
import yfinance as yf
from signum import Chart

# Download 1 year of AAPL data
df = yf.download("AAPL", period="1y", auto_adjust=True)
df = df.reset_index()
df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]  # flatten multi-index

# Calculate moving averages
sma_20 = pd.DataFrame({"time": df["Date"], "value": df["Close"].rolling(20).mean()}).dropna()
sma_50 = pd.DataFrame({"time": df["Date"], "value": df["Close"].rolling(50).mean()}).dropna()

print(f"Loaded {len(df)} bars for AAPL")
df.tail(3)

[*********************100%***********************]  1 of 1 completed

Loaded 251 bars for AAPL


,Date,Close,High,Low,Open,Volume
248,2026-03-13,250.119995,256.329987,249.520004,255.479996,36930000
249,2026-03-16,252.820007,253.889999,249.880005,252.110001,32074200
250,2026-03-17,254.229996,255.129898,252.179993,253.078506,27556024


## 1. Candlestick Chart with Volume
Basic OHLCV chart rendered inline in the notebook.

In [2]:
# Dark theme candlestick + volume overlay
chart = Chart(theme="light", height=450)
chart.candlestick(df).volume(df)
chart

## 2. With Moving Average Overlays
Chain `.candlestick()`, `.line()`, `.volume()` to compose the chart. Each `.line()` auto-cycles colors.

In [4]:
# Midnight theme with SMA overlays + watermark
chart = (
    Chart(theme="midnight", height=500, watermark="AAPL")
    .candlestick(df)
    .line(sma_20, name="SMA 20", color="#FF6D00", width=1)
    .line(sma_50, name="SMA 50", color="#E91E63", width=1)
    .volume(df)
)
chart

## 3. Area Chart (Equity Curve Style)
Great for portfolio value, NAV, or cumulative returns.

In [5]:
# Area chart using close prices
Chart(theme="dark", height=350).area(df, value_col="Close", name="AAPL")

## 4. Baseline Chart (P&L / Returns)
Green above zero, red below — ideal for showing cumulative returns or P&L relative to a baseline.

In [6]:
# Cumulative returns as baseline chart
returns = pd.DataFrame({
    "time": df["Date"],
    "value": ((df["Close"] / df["Close"].iloc[0]) - 1) * 100
}).dropna()

Chart(theme="dark", height=350).baseline(returns, base_value=0, value_col="value")

## 5. Financial Times Theme
Matching ForgeFolio's FT theme — cream background, Georgia serif, burgundy/teal palette.

In [7]:
# Financial Times theme
Chart(theme="ft", height=400, watermark="AAPL").candlestick(df).volume(df)

## 6. Save & Integration

The **same chart object** works everywhere:

| Method | Where |
|--------|-------|
| `chart` or `chart.show()` | Jupyter Notebook (inline iframe) |
| `chart.save("out.html")` | Standalone HTML file |
| `chart.to_dash(id="x")` | Dash app component (`html.Iframe`) |
| `chart.to_streamlit()` | Streamlit dashboard |
| `chart.render()` | Raw HTML string for any embedding |

In [8]:
# Save as standalone HTML
chart = Chart(theme="dark", watermark="AAPL").candlestick(df).volume(df)
chart.save("aapl_chart.html")
print("✅ Saved to aapl_chart.html — open in any browser\n")

# Dash integration (copy-paste into a .py file):
print("""# ── Dash App ──────────────────────────────────
from dash import Dash, html
from signum import Chart

chart = Chart(theme="dark").candlestick(df).volume(df)

app = Dash(__name__)
app.layout = html.Div([
    html.H1("AAPL Dashboard"),
    chart.to_dash(id="price-chart"),
])
app.run(debug=True)
""")

# Streamlit integration (copy-paste into a .py file):
print("""# ── Streamlit App ─────────────────────────────
import streamlit as st
from signum import Chart

chart = Chart(theme="dark").candlestick(df).volume(df)
st.title("AAPL Dashboard")
chart.to_streamlit()
""")

✅ Saved to aapl_chart.html — open in any browser

# ── Dash App ──────────────────────────────────
from dash import Dash, html
from signum import Chart

chart = Chart(theme="dark").candlestick(df).volume(df)

app = Dash(__name__)
app.layout = html.Div([
    html.H1("AAPL Dashboard"),
    chart.to_dash(id="price-chart"),
])
app.run(debug=True)

# ── Streamlit App ─────────────────────────────
import streamlit as st
from signum import Chart

chart = Chart(theme="dark").candlestick(df).volume(df)
st.title("AAPL Dashboard")
chart.to_streamlit()



## 7. Price Lines & Markers
Add horizontal price lines and markers for signals, targets, support/resistance levels.

In [9]:
# Annotated chart with price lines and signal markers
avg_price = float(df["Close"].mean())
max_price = float(df["Close"].max())
min_date = df.loc[df["Close"].idxmin(), "Date"].strftime("%Y-%m-%d")
max_date = df.loc[df["Close"].idxmax(), "Date"].strftime("%Y-%m-%d")

chart = (
    Chart(theme="midnight", height=450, watermark="AAPL")
    .candlestick(df)
    .volume(df)
    .price_line(avg_price, title="Average", color="#FF9800")
    .marker(min_date, text="Low", position="belowBar", shape="arrowUp", color="#26a69a")
    .marker(max_date, text="High", position="aboveBar", shape="arrowDown", color="#ef5350")
)
chart

## 8. Buy / Sell Signals with Position Shading
Buy/sell arrows via `.signals()` plus `.shade()` to highlight periods when the strategy is "in the deal". The shaded background makes it easy to see at a glance when you're invested.

In [10]:
import importlib, signum.engine.chart
importlib.reload(signum.engine.chart)
from signum.engine.chart import Chart
import numpy as np

# ── Artificial buy/sell signals at fixed dates ────────────────────
# (In practice these come from your model — here we just pick dates)
buy_dates  = df["Date"].iloc[np.linspace(30, len(df)-60, 4, dtype=int)].values
sell_dates = df["Date"].iloc[np.linspace(60, len(df)-30, 4, dtype=int)].values

df["signal"] = 0
df.loc[df["Date"].isin(buy_dates), "signal"] = 1
df.loc[df["Date"].isin(sell_dates), "signal"] = -1

# Build position column: 1 between buy and sell, 0 otherwise
df["position"] = 0
in_pos = False
for i in range(len(df)):
    if df["signal"].iloc[i] == 1:
        in_pos = True
    elif df["signal"].iloc[i] == -1:
        in_pos = False
    df.iloc[i, df.columns.get_loc("position")] = 1 if in_pos else 0

n_buys = (df["signal"] == 1).sum()
n_sells = (df["signal"] == -1).sum()
print(f"{n_buys} BUY / {n_sells} SELL signals  •  In market {df['position'].mean()*100:.0f}% of the time")

# ── Chart with arrows + shaded background during position ─────────
chart = (
    Chart(theme="dark", height=500, watermark="AAPL")
    .candlestick(df)
    .volume(df)
    .signals(df, signal_col="signal")
    .shade(df, position_col="position")
)
chart

4 BUY / 4 SELL signals  •  In market 48% of the time


## 9. Multi-Pane Synchronized Dashboard
Vertically stacked charts with synchronized crosshair, zoom, and scroll.

Pattern inspired by TKAN v2 (Price → Equity → IVol) and Cogilator (Price → Signal → Position).

In [11]:
import importlib, signum.engine.chart, signum.engine.dashboard, signum
importlib.reload(signum.engine.chart)
importlib.reload(signum.engine.dashboard)
from signum.engine.chart import Chart
from signum.engine.dashboard import Dashboard

import numpy as np

# ── Simulate signal, position, equity from price data ─────────────
sma20 = df["Close"].rolling(20).mean()
sma50 = df["Close"].rolling(50).mean()

# Momentum signal: SMA20 vs SMA50 spread (normalised)
signal = ((sma20 - sma50) / sma50 * 100)

# Position: +1 when SMA20 > SMA50, else 0
pos = (sma20 > sma50).astype(float)

# Simple equity: cumulative returns when in position
daily_ret = df["Close"].pct_change()
strat_ret = daily_ret * pos.shift(1)
equity = (1 + strat_ret.fillna(0)).cumprod() * 100

# ── Align all data to the same date range ─────────────────────────
# Find first date where ALL series have data (after the 50-day SMA warm-up)
mask = signal.notna() & pos.notna() & equity.notna()
df_aligned = df[mask].copy()
sig_df = pd.DataFrame({"time": df_aligned["Date"], "value": signal[mask].values})

# Re-index equity to start at 100 from aligned start
eq_vals = equity[mask].values
eq_vals = eq_vals / eq_vals[0] * 100
eq_df = pd.DataFrame({"time": df_aligned["Date"], "value": eq_vals})
sma_20_aligned = sma_20[sma_20["time"].isin(df_aligned["Date"])]
sma_50_aligned = sma_50[sma_50["time"].isin(df_aligned["Date"])]

print(f"Aligned: {len(df_aligned)} bars from {df_aligned['Date'].iloc[0].date()} to {df_aligned['Date'].iloc[-1].date()}")

# ── Create 3-pane synchronized dashboard ──────────────────────────
price_pane = (
    Chart(height=300)
    .candlestick(df_aligned)
    .line(sma_20_aligned, name="SMA 20", color="#FF6D00", width=1)
    .line(sma_50_aligned, name="SMA 50", color="#E91E63", width=1)
    .volume(df_aligned)
)
signal_pane = Chart(height=150).baseline(sig_df, base_value=0)
equity_pane = Chart(height=180).area(eq_df, name="Strategy Equity")

dash = Dashboard(
    panes=[price_pane, signal_pane, equity_pane],
    titles=["AAPL Price + Moving Averages", "Momentum Signal (SMA Spread %)", "Equity Curve (indexed 100)"],
)
dash

Aligned: 202 bars from 2025-05-28 to 2026-03-17


## 10. Time in Market — Signal Dashboard
Multi-pane dashboard with shaded position on price, continuous signal evolution, and equity comparison.

| Pane | Content |
|------|---------|
| **Price** | Candlestick + Buy/Sell arrows + shaded background when in position |
| **Signal** | Continuous momentum signal (SMA spread %) — tracks signal buildup and decay |
| **Position** | Binary 1/0 histogram — time in market at a glance |
| **Equity** | Strategy vs Buy & Hold (indexed 100) |

In [12]:
import importlib, signum.engine.chart, signum.engine.dashboard, signum
importlib.reload(signum.engine.chart)
importlib.reload(signum.engine.dashboard)
from signum.engine.chart import Chart
from signum.engine.dashboard import Dashboard

# ── Compute continuous signal + position from SMA crossover ───────
sma20 = df["Close"].rolling(20).mean()
sma50 = df["Close"].rolling(50).mean()

# Continuous signal: SMA spread % (tracks signal buildup/decay)
signal_vals = ((sma20 - sma50) / sma50 * 100)

# Position: +1 when SMA20 > SMA50, else 0
position = (sma20 > sma50).astype(float)

# Equity curves
daily_ret = df["Close"].pct_change()
strat_ret = daily_ret * position.shift(1)
strategy_eq = (1 + strat_ret.fillna(0)).cumprod() * 100
bh_eq = (1 + daily_ret.fillna(0)).cumprod() * 100

# ── Align to common date range ────────────────────────────────────
mask = signal_vals.notna() & position.notna()
dates = df.loc[mask, "Date"]
df_aligned = df[mask].copy()
df_aligned["position"] = position[mask].values

# Re-index both equity curves to start at 100 from the aligned start
strat_aligned = strategy_eq[mask].values
bh_aligned = bh_eq[mask].values
strat_aligned = strat_aligned / strat_aligned[0] * 100
bh_aligned = bh_aligned / bh_aligned[0] * 100

sig_df  = pd.DataFrame({"time": dates, "value": signal_vals[mask].values})
pos_df  = pd.DataFrame({"time": dates, "value": position[mask].values})
eq_df   = pd.DataFrame({"time": dates, "value": strat_aligned})
bh_df   = pd.DataFrame({"time": dates, "value": bh_aligned})

# Generate crossover signals for markers
cross_above = (sma20[mask].values > sma50[mask].values) & \
              (pd.Series(sma20[mask].values).shift(1).values <= pd.Series(sma50[mask].values).shift(1).values)
cross_below = (sma20[mask].values < sma50[mask].values) & \
              (pd.Series(sma20[mask].values).shift(1).values >= pd.Series(sma50[mask].values).shift(1).values)
df_aligned["signal"] = 0
df_aligned.loc[df_aligned.index[cross_above], "signal"] = 1
df_aligned.loc[df_aligned.index[cross_below], "signal"] = -1

# ── Stats ─────────────────────────────────────────────────────────
time_in_market = position[mask].mean() * 100
print(f"Time in Market: {time_in_market:.1f}%  ({int(position[mask].sum())}/{len(df_aligned)} days)")
print(f"Strategy Return: {strategy_eq[mask].iloc[-1] - 100:+.1f}%  |  Buy & Hold: {bh_eq[mask].iloc[-1] - 100:+.1f}%")

# ── Build 4-pane dashboard ────────────────────────────────────────
price_pane = (
    Chart(height=280)
    .candlestick(df_aligned)
    .volume(df_aligned)
    .signals(df_aligned, signal_col="signal")
    .shade(df_aligned, position_col="position")
)

signal_pane = Chart(height=120).baseline(sig_df, base_value=0)

position_pane = (
    Chart(height=90)
    .histogram(pos_df, name="Position", color="rgba(38, 166, 154, 0.7)")
)

equity_pane = (
    Chart(height=170)
    .area(eq_df, name="Strategy", color="#26a69a")
    .line(bh_df, name="Buy & Hold", color="#5c6bc0", width=1)
)

dash = Dashboard(
    panes=[price_pane, signal_pane, position_pane, equity_pane],
    titles=[
        "AAPL — Price + Position Shading",
        "Signal (SMA Spread %) — continuous evolution",
        f"Position (Time in Market: {time_in_market:.0f}%)",
        "Equity — Strategy vs Buy & Hold (indexed 100)",
    ],
)
dash

Time in Market: 73.8%  (149/202 days)
Strategy Return: +15.1%  |  Buy & Hold: +20.1%
